# CS781 Capstone · Phase 1
## Decoder-only model evaluation

This notebook evaluates the 217,860,096-parameter decoder trained from scratch in
`decoder_004_mixed217m`. It restores `best_model.pt`, which was selected by held-out language
modeling NLL, and compares decoding settings on the frozen `prep_002_mixed16k` generation split.

| Component | Configuration |
|---|---|
| Model | 16 pre-norm causal blocks, width 1,024, 16 attention heads, 4,096-dimensional feed-forward layers |
| Token representation | Shared 16,000-ID byte-BPE embeddings and output weights, sinusoidal positions |
| Context | Up to 2,560 tokens |
| Validation | 100 balanced examples, 50 original Supreme Court anchors, and per-source slices |
| Scoring | Course evaluator with BERTScore F1 as the primary metric, plus ROUGE-L F1 and BLEU-4 |
| Submission format | A ZIP with root-level `prediction.json`, keyed by the 1,000 official prompt IDs |

Each continuation targets 128 words. The same model checkpoint and held-out references are
used across decoding settings. The scorer loads a pretrained encoder for BERTScore.
The generator uses the Phase 1 training weights.


## 1. Kaggle configuration

1. Select a GPU accelerator. A single T4 is sufficient for the default inference path.
2. Attach the extracted `prep_002_mixed16k` and `decoder_004_mixed217m` folders as private inputs.
   The decoder folder must contain `best_model.pt`. Use only trusted backups.
3. Enable Internet for the first BERTScore model download and to install `bert-score` if needed.
4. Run all cells. Set the folder overrides below if there are multiple copies or a renamed directory.

The default compares greedy, top-p 0.95 at temperature 0.1, and top-p 0.95 at temperature 1.0.
For a shorter validation run, set `RUN_DECODING_NAMES = ("top_p_095_t1",)`. To generate the
official test after reviewing validation, set `GENERATE_OFFICIAL_TEST = True` and choose
`TEST_DECODING_NAME`, then rerun the configuration and downstream cells. Cached validation
work is reused. Keep the model weights, prepared data, and run backups outside GitHub.


In [1]:
from pathlib import Path

PREP_INPUT = None            # Inner folder containing cache_manifest.json
RUN_INPUT = None             # Inner folder containing best_model.pt
EVAL_RESUME_INPUT = None     # Optional extracted evaluation folder containing evaluation_protocol.json

EVALUATION_TAG = "v1"
GENERATION_BATCH_SIZE = 4
MAX_NEW_TOKENS = 1024
GENERATION_SEED = 456
RUN_DECODING_NAMES = ("greedy", "top_p_095_t01", "top_p_095_t1")

GENERATE_OFFICIAL_TEST = False
TEST_DECODING_NAME = "top_p_095_t1"
CREATE_EVALUATION_ZIP = True

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / "phase1_decoder_evaluation_outputs"
PREPARATION_ID = "prep_002_mixed16k"
RUN_ID = "decoder_004_mixed217m"
SOURCE_ORDER = ["supreme_court", "high_courts", "acts", "tribunals"]
DECODING_SETTINGS = {
    "greedy": {"mode": "greedy"},
    "top_p_095_t01": {"mode": "top_p", "top_p": 0.95, "temperature": 0.1},
    "top_p_095_t1": {"mode": "top_p", "top_p": 0.95, "temperature": 1.0},
    "top_p_095_t07": {"mode": "top_p", "top_p": 0.95, "temperature": 0.7},
    "top_k_50_t1": {"mode": "top_k", "top_k": 50, "temperature": 1.0},
}
if not RUN_DECODING_NAMES or set(RUN_DECODING_NAMES) - DECODING_SETTINGS.keys():
    raise ValueError("Choose at least one valid decoding name.")
if len(set(RUN_DECODING_NAMES)) != len(RUN_DECODING_NAMES):
    raise ValueError("List each decoding setting once.")
if GENERATE_OFFICIAL_TEST and TEST_DECODING_NAME not in DECODING_SETTINGS:
    raise ValueError("Choose a valid official test decoding name.")
if not EVALUATION_TAG or not EVALUATION_TAG.replace("_", "").replace("-", "").isalnum():
    raise ValueError("Use letters, digits, underscores or hyphens for EVALUATION_TAG.")
if GENERATION_BATCH_SIZE < 1 or MAX_NEW_TOKENS < 1:
    raise ValueError("Batch size and token cap must be positive.")
print(f"Decoder validation · {', '.join(RUN_DECODING_NAMES)}", flush=True)


Decoder validation · greedy, top_p_095_t01, top_p_095_t1


## 2. Project source and dependencies

The notebook carries the original tokenizer and decoder definitions plus the course evaluator.
The generation code retains fixed per-prompt seeds, batched causal decoding and atomic
progress writes. The evaluation outputs are stored in the working directory.


In [2]:
from pathlib import Path
from importlib.metadata import version
import gc
import json
import os
import pickle
import platform
import shutil
import sys
from tempfile import TemporaryDirectory
import time

import pandas as pd
from IPython.display import FileLink, display

from importlib.util import find_spec
from IPython import get_ipython
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator before running the decoder evaluation.")
print(f"PyTorch {torch.__version__} · {torch.cuda.get_device_name(0)}", flush=True)
if find_spec("bert_score") is None:
    print("Installing the course evaluator dependency", flush=True)
    get_ipython().run_line_magic("pip", "install -q bert-score==0.3.13")


PyTorch 2.11.0+cu130 · Tesla T4
Installing the course evaluator dependency
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Project source carried by this notebook for fresh sessions.
modules = {
    'phase1_utils.py': r'''"""Small file and JSON helpers used by the Phase-1 notebooks."""
import hashlib
import json
import os
import pickle
import zipfile
from pathlib import Path


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def atomic_pickle(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def checked_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != value:
        raise ValueError(f"Saved settings differ: {path}. Use a new experiment ID.")
    atomic_json(path, value)


def write_source(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw = content.encode("utf-8") if isinstance(content, str) else content
    if path.exists():
        if path.read_bytes() != raw:
            raise ValueError(f"Saved source differs: {path}. Keep its code or use a new experiment ID.")
        return
    path.write_bytes(raw)


def make_bundle(folder, output_path, archive_root):
    folder, output_path, archive_root = Path(folder), Path(output_path), Path(archive_root)
    temporary = output_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(archive_root))
    os.replace(temporary, output_path)
    return output_path
''',
    'a1_tokenizer.py': r'''"""The assignment byte-BPE algorithm, with checkpoints between merges.

Pair counting, lexicographic tie breaking, non-overlapping replacements and
merge-order encoding are unchanged from the completed Phase-1/A3 tokenizer.
"""
import pickle
import re
import time
from collections import Counter
from pathlib import Path
from phase1_utils import atomic_pickle, sha256_file, text_hash


class A1BPETokenizer:
    def __init__(self):
        specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.vocab = [s.encode("utf-8") for s in specials]
        self.vocab += [bytes([b]) for b in range(256)]
        self.boundary_id = len(self.vocab)
        self.vocab.append(b"")
        self.names = [s.encode("utf-8") for s in specials]
        self.names += [bytes([b]) for b in range(256)]
        self.names.append(b"</w>")
        self.merges = {}
        self.vocab_size = len(self.vocab)

    def fit(self, text, vocab_size, checkpoint_path=None, checkpoint_merges=100,
            stop_after_merges=None):
        if vocab_size < 261:
            raise ValueError("Vocabulary must accommodate bytes and reserved tokens")
        signature = {"text_sha256": text_hash(text), "vocab_size": vocab_size,
                     "algorithm_source_sha256": sha256_file(__file__)}
        checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        elapsed_before = 0.0
        if checkpoint_path is not None and checkpoint_path.exists():
            with open(checkpoint_path, "rb") as file:
                saved = pickle.load(file)
            if saved["signature"] != signature:
                raise ValueError("BPE checkpoint settings or source differ")
            self.__dict__.update(saved["tokenizer_state"])
            words = saved["words"]
            elapsed_before = saved["elapsed_seconds"]
            print("Resuming BPE vocabulary:", self.vocab_size, flush=True)
        else:
            if self.merges:
                raise ValueError("Use a fresh tokenizer or its saved fitting checkpoint")
            words = Counter(tuple(b + 4 for b in piece.encode("utf-8")) + (self.boundary_id,)
                            for piece in re.findall(r"\S+|\s+", text))
        started = time.monotonic()
        last_saved = started
        merges_this_call = 0
        commit_in_progress = False

        def snapshot():
            if checkpoint_path is not None:
                atomic_pickle(checkpoint_path, {"signature": signature,
                    "tokenizer_state": self.__dict__, "words": words,
                    "elapsed_seconds": elapsed_before + time.monotonic() - started})

        try:
            while len(self.vocab) < vocab_size:
                pair_counts = Counter()
                for symbols, frequency in words.items():
                    for pair in zip(symbols, symbols[1:]):
                        pair_counts[pair] += frequency
                if not pair_counts:
                    break
                pair = min(pair_counts, key=lambda p: (-pair_counts[p], self.names[p[0]], self.names[p[1]]))
                if pair_counts[pair] < 2:
                    break
                new_id = len(self.vocab)
                updated = Counter()
                for symbols, frequency in words.items():
                    result = []
                    j = 0
                    while j < len(symbols):
                        if j + 1 < len(symbols) and symbols[j:j + 2] == pair:
                            result.append(new_id)
                            j += 2
                        else:
                            result.append(symbols[j])
                            j += 1
                    updated[tuple(result)] += frequency
                # Commit the whole merge only after every word has been replaced.
                commit_in_progress = True
                self.vocab.append(self.vocab[pair[0]] + self.vocab[pair[1]])
                self.names.append(self.names[pair[0]] + self.names[pair[1]])
                self.merges[pair] = new_id
                self.vocab_size = len(self.vocab)
                words = updated
                commit_in_progress = False
                merges_this_call += 1
                if len(self.merges) % checkpoint_merges == 0 or time.monotonic() - last_saved >= 60:
                    snapshot()
                    print(f"BPE vocabulary {self.vocab_size:,}/{vocab_size:,} | "
                          f"elapsed {(elapsed_before + time.monotonic() - started)/60:.1f} min", flush=True)
                    last_saved = time.monotonic()
                if stop_after_merges is not None and merges_this_call >= stop_after_merges:
                    break
        finally:
            if not commit_in_progress:
                snapshot()
        return self

    def encode(self, text):
        result = []
        for piece in re.findall(r"\S+|\s+", text):
            symbols = [b + 4 for b in piece.encode("utf-8")] + [self.boundary_id]
            while len(symbols) > 1:
                candidates = [(self.merges[(a, b)], (a, b))
                              for a, b in zip(symbols, symbols[1:]) if (a, b) in self.merges]
                if not candidates:
                    break
                new_id, pair = min(candidates)
                merged = []
                j = 0
                while j < len(symbols):
                    if j + 1 < len(symbols) and (symbols[j], symbols[j + 1]) == pair:
                        merged.append(new_id)
                        j += 2
                    else:
                        merged.append(symbols[j])
                        j += 1
                symbols = merged
            result.extend(symbols)
        return result

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")
''',
    'phase1_decoder.py': r'''"""Phase 1 decoder, adapted from completed Assignment 3 Q1."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int = 16000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    d_ff: int = 4096
    max_seq_len: int = 2560
    dropout: float = 0.1
    norm_first: bool = True
    attention_impl: str = "sdpa"
    attention_scale: float = None
    embedding_scale: float = 32.0


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0,
                 attention_scale=None, attention_impl="sdpa"):
        super().__init__()
        assert d_model % n_heads == 0
        if attention_impl not in ("manual", "sdpa"):
            raise ValueError("attention_impl must be manual or sdpa")
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head) if attention_scale is None else float(attention_scale)
        self.attention_impl = attention_impl
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if attention_impl == "manual":
            self.register_buffer("causal_mask", torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)),
                                 persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        if self.attention_impl == "manual":
            scores = (q @ k.transpose(-2, -1)) * self.attention_scale
            scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
            weights = self.attn_dropout(torch.softmax(scores, dim=-1))
            out = weights @ v
        else:
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True, scale=self.attention_scale,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
            )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    position = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * frequency)
    pe[:, 1::2] = torch.cos(position * frequency)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                      cfg.attention_scale, cfg.attention_impl)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model),
                             persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

        # parameters() counts the tied matrix once.
        for parameter in self.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        for module in self.modules():
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, last_positions=None):
        B, T = idx.shape
        if T > self.cfg.max_seq_len:
            raise ValueError("Input exceeds the configured context length")
        x = self.tok_emb(idx) * self.cfg.embedding_scale + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        if last_positions is not None:
            x = x[torch.arange(B, device=idx.device), last_positions]
        return self.head(x)


class ModelWithLoss(nn.Module):
    """Compute loss on each GPU before DataParallel gathers the small outputs."""
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm = lm
        self.use_amp = use_amp

    def forward(self, x, y):
        # Enter autocast in each worker thread; do not move the model to half().
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits = self.lm(x)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), y.reshape(-1))
        return loss.unsqueeze(0)
''',
    'phase1_generation.py': r'''"""Manual batched autoregressive decoding; no external generation model."""
import json
import time
from pathlib import Path
import torch
from phase1_utils import atomic_json, read_json, checked_json, text_hash


def sample_next(logits, mode, temperature, top_k, top_p, generator):
    if mode == "greedy":
        return int(logits.argmax().item())
    scaled = logits.float() / temperature
    if mode == "top_k":
        values, indices = scaled.topk(min(top_k, scaled.numel()))
        position = torch.multinomial(torch.softmax(values, dim=-1), 1, generator=generator)
        return int(indices[position].item())
    if mode == "top_p":
        values, indices = scaled.sort(descending=True)
        probabilities = torch.softmax(values, dim=-1)
        # Retain the first token that takes cumulative probability over p.
        remove = probabilities.cumsum(-1) - probabilities >= top_p
        values[remove] = -float("inf")
        position = torch.multinomial(torch.softmax(values, dim=-1), 1, generator=generator)
        return int(indices[position].item())
    raise ValueError("Unknown decoding mode")


def decoded_text(tokenizer, ids):
    # Byte BPE can temporarily end partway through a UTF-8 character.
    raw = b"".join(tokenizer.vocab[i] for i in ids if i >= 4)
    return raw.decode("utf-8", errors="replace")


@torch.no_grad()
def generate_predictions(model, tokenizer, prompts, output_dir, protocol, batch_size=4,
                         max_new_tokens=1024, target_words=128, seed=1234):
    """Save after complete fixed batches, so interrupted groups can be redone."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()
    ids = list(prompts)
    frozen = {**protocol, "prompt_sha256": text_hash(json.dumps(prompts, sort_keys=True)),
              "prompt_id_order": ids,
              "batch_size": batch_size, "max_new_tokens": max_new_tokens,
              "target_words": target_words, "seed": seed, "context": model.cfg.max_seq_len,
              "torch_version": str(torch.__version__)}
    checked_json(output_dir / "protocol.json", frozen)
    progress_path = output_dir / "progress.json"
    progress = read_json(progress_path) if progress_path.exists() else {"rows": {}}
    rows = progress["rows"]
    if set(rows) - set(ids):
        raise ValueError("Generation progress contains unknown IDs")
    started = time.monotonic()
    last_report = started
    print(f"Generation · {len(rows):,}/{len(ids):,} saved prompts · batch size {batch_size}", flush=True)
    for start in range(0, len(ids), batch_size):
        group = ids[start:start + batch_size]
        done = [key in rows for key in group]
        if all(done):
            continue
        if any(done):
            raise ValueError("Partial batch in saved progress; expected atomic group commits")
        contexts = [tokenizer.encode(prompts[key]["text"]) for key in group]
        generated = [[] for _ in group]
        finished = [False] * len(group)
        stop_reasons = ["token_cap"] * len(group)
        generators = [torch.Generator(device=device).manual_seed(seed + int(text_hash(key)[:8], 16))
                      for key in group]
        original_lengths = [len(context) for context in contexts]
        cropped = [max(0, length - model.cfg.max_seq_len) for length in original_lengths]
        for step in range(1, max_new_tokens + 1):
            active = [j for j in range(len(group)) if not finished[j]]
            if not active:
                break
            windows = [contexts[j][-model.cfg.max_seq_len:] for j in active]
            if any(not window for window in windows):
                raise ValueError("Empty tokenized prompt")
            lengths = torch.tensor([len(window) for window in windows], device=device)
            x = torch.zeros((len(active), int(lengths.max().item())), dtype=torch.long, device=device)
            for k, window in enumerate(windows):
                x[k, :len(window)] = torch.tensor(window, device=device)
            # Right padding occurs strictly AFTER the real last position.
            # Causal attention prevents it affecting any real-position output.
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(x, last_positions=lengths - 1).float()
            for k, j in enumerate(active):
                logits[k, :3] = -float("inf")  # never emit PAD / UNK / BOS
                text = decoded_text(tokenizer, generated[j])
                if len(text.split()) < target_words:
                    logits[k, 3] = -float("inf")
                token = sample_next(logits[k], protocol["mode"], protocol.get("temperature", 1.0),
                                    protocol.get("top_k", 50), protocol.get("top_p", 0.95), generators[j])
                if token == 3:
                    finished[j], stop_reasons[j] = True, "eos_after_word_target"
                    continue
                generated[j].append(token)
                contexts[j].append(token)
                cropped[j] += int(len(contexts[j]) > model.cfg.max_seq_len)
                text = decoded_text(tokenizer, generated[j])
                # Wait for the 128th word to be complete: whitespace or a
                # subsequent word. At the cap, a final word may be unfinished.
                if len(text.split()) > target_words or (len(text.split()) >= target_words and text[-1:].isspace()):
                    finished[j], stop_reasons[j] = True, "word_target"
            now = time.monotonic()
            if now - last_report >= 60:
                print(f"Generation · {len(rows):,}/{len(ids):,} saved · batch at token step {step} "
                      f"· {(now - started) / 60:.1f} min", flush=True)
                last_report = now
        for j, key in enumerate(group):
            raw = decoded_text(tokenizer, generated[j])
            words = raw.split()
            text = " ".join(words[:target_words])
            rows[key] = {"text": text, "generated_tokens": len(generated[j]),
                         "word_count": len(text.split()), "stop_reason": stop_reasons[j],
                         "prompt_tokens": original_lengths[j],
                         "prompt_tokens_dropped": max(0, original_lengths[j] - model.cfg.max_seq_len),
                         "rolling_context_steps": cropped[j] - max(0, original_lengths[j] - model.cfg.max_seq_len),
                         "contains_utf8_replacement": "\ufffd" in text,
                         "complete": stop_reasons[j] != "token_cap" and len(words) >= target_words}
        atomic_json(progress_path, {"rows": rows})
        now = time.monotonic()
        if start == 0 or len(rows) == len(ids) or now - last_report >= 60:
            print(f"Generation · {len(rows):,}/{len(ids):,} saved · {(now - started) / 60:.1f} min", flush=True)
            last_report = now
    predictions = {key: {"text": rows[key]["text"]} for key in ids}
    atomic_json(output_dir / "prediction.json", predictions)
    incomplete = [key for key in ids if not rows[key]["complete"]]
    atomic_json(output_dir / "summary.json", {"prompts": len(ids), "incomplete_ids": incomplete,
        "elapsed_seconds_this_invocation": time.monotonic() - started,
        "cropped_prompt_count": sum(rows[key]["prompt_tokens"] > model.cfg.max_seq_len for key in ids),
        "rolling_context_used_count": sum(rows[key]["rolling_context_steps"] > 0 for key in ids),
        "mean_generated_tokens": sum(rows[key]["generated_tokens"] for key in ids) / len(ids)})
    return predictions, rows
''',
    'evaluate_phase1.py': r'''#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 1 predictions on your own held-out split.')
    ap.add_argument('--truth', required=True, help='Your held-out reference JSON.')
    ap.add_argument('--prediction', required=True, help='Your prediction JSON.')
    args=ap.parse_args()
    truth=load_json(args.truth); pred=load_json(args.prediction)
    validate(truth,'truth'); validate(pred,'prediction')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    ps=[pred[i]['text'].split() for i in truth]; rs=[truth[i]['text'].split() for i in truth]
    pred_texts=[pred[i]['text'].strip() for i in truth]; ref_texts=[truth[i]['text'].strip() for i in truth]
    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore=float(f.mean().item())
    rouge=sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in truth)/len(truth)
    b4=bleu4(ps,rs)
    print(f'Examples: {len(truth)}')
    print(f'BERTScore F1: {bertscore:.6f}')
    print(f'ROUGE-L F1:   {rouge:.6f}')
    print(f'BLEU-4:       {b4:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}', file=sys.stderr); sys.exit(1)
''',
}
SRC_DIR = OUTPUT_ROOT / '_source'
SRC_DIR.mkdir(parents=True, exist_ok=True)
for name, source in modules.items():
    path = SRC_DIR / name
    if not path.exists() or path.read_bytes() != source.encode('utf-8'):
        path.write_bytes(source.encode('utf-8'))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [4]:
from phase1_utils import atomic_json, checked_json, make_bundle, read_json, sha256_file
from phase1_decoder import DecoderLM, LMConfig
from phase1_generation import generate_predictions
from a1_tokenizer import A1BPETokenizer

def resolve_input(explicit, names, marker, setting, *, required=True):
    """Prefer an explicit folder or a matching folder in the work area."""
    if explicit is not None:
        folder = Path(explicit).expanduser().resolve()
        if not (folder / marker).is_file():
            raise FileNotFoundError(f"{setting} must point to the folder containing {marker}: {folder}")
        return folder
    for root in (WORK_ROOT / "cs781_phase1", INPUT_ROOT):
        matches = sorted({
            path.parent.resolve()
            for path in root.rglob(marker)
            if path.parent.name in names and path.is_file()
        })
        if len(matches) > 1:
            paths = "\n".join(map(str, matches))
            raise ValueError(f"Set {setting} to one of these folders:\n{paths}")
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(
            f"Attach an extracted {', '.join(names)} folder containing {marker}, or set {setting}."
        )
    return None


def restore_folder(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=destination.parent) as temporary:
        staged = Path(temporary) / "backup"
        shutil.copytree(source, staged)
        staged.rename(destination)


def show_download(path):
    display(FileLink(os.path.relpath(path, Path.cwd())))

def load_validation(prep_dir, sources):
    prompts = read_json(prep_dir / "validation_prompts.json")
    truths = read_json(prep_dir / "validation_truth.json")
    groups = read_json(prep_dir / "validation_groups.json")
    anchor = groups["anchor_50"]
    balanced = groups["balanced_100"]
    per_source = groups["balanced_source_ids"]
    if (
        len(prompts) != 125 or set(prompts) != set(truths)
        or len(anchor) != 50 or len(set(anchor)) != 50
        or len(balanced) != 100 or len(set(balanced)) != 100
        or set(per_source) != set(sources)
        or any(len(per_source[source]) != 25 for source in sources)
        or [key for source in sources for key in per_source[source]] != balanced
        or per_source["supreme_court"] != anchor[:25]
        or set(anchor) | set(balanced) != set(prompts)
    ):
        raise ValueError("The saved anchor, balanced, and per-source groups do not match the Phase 1 split.")
    for key in prompts:
        if len(prompts[key]["text"].split()) != 512 or len(truths[key]["text"].split()) != 128:
            raise ValueError(f"Unexpected prompt/reference length for {key}.")
    return prompts, truths, groups


def load_test_prompts(prep_dir):
    prompts = read_json(prep_dir / "official_test_prompts.json")
    if not isinstance(prompts, dict) or len(prompts) != 1000:
        raise ValueError("Expected 1,000 official prompt IDs.")
    if any(not isinstance(row, dict) or not isinstance(row.get("text"), str) or not row["text"].strip()
        for row in prompts.values()):
        raise ValueError("The official prompt file contains an empty or invalid text field.")
    return prompts


## 3. Verify inputs and pin the evaluation

Confirm the preparation and decoder identities before loading the model. Save the
checkpoint hash, prompt and reference hashes, runtime, and generation settings alongside
the results. Each output folder is tied to a checkpoint hash. Use a new `EVALUATION_TAG`
when changing batch size, token cap, seed or GPU environment.

A session restart can reuse the working files. In a fresh session, attach the extracted
evaluation backup and point `EVAL_RESUME_INPUT` to its inner folder. Use matching settings
and the same GPU type for deterministic resume.


In [5]:
from bert_score.utils import lang2model, model2layers

PREP_DIR = resolve_input(PREP_INPUT, (PREPARATION_ID,), "cache_manifest.json", "PREP_INPUT")
RUN_DIR = resolve_input(RUN_INPUT, (RUN_ID, "decoder_004_mixed_217m"), "best_model.pt", "RUN_INPUT")
print(f"Preparation folder: {PREP_DIR}")
print(f"Decoder folder: {RUN_DIR}")
manifest = read_json(PREP_DIR / "cache_manifest.json")
tokenizer_signature = manifest["tokenizer_signature"]
tokenizer_path = PREP_DIR / "tokenizer_state.pkl"
best_path = RUN_DIR / "best_model.pt"
decoder_source = SRC_DIR / "phase1_decoder.py"
known_decoder_sha = "fd5add7ccc1d519a6f547fbd9a5c86a53276db8e09fb93c165d96531685d0bb7"

if manifest["preparation_id"] != PREPARATION_ID or manifest["source_order"] != SOURCE_ORDER:
    raise ValueError("Attach the shared Phase 1 mixed preparation.")
if sha256_file(tokenizer_path) != manifest["tokenizer_state_sha256"]:
    raise ValueError("The tokenizer file differs from the preparation manifest.")
if sha256_file(SRC_DIR / "a1_tokenizer.py") != tokenizer_signature["algorithm_source_sha256"]:
    raise ValueError("The notebook tokenizer implementation differs from the preparation.")
print("Hashing the saved decoder checkpoint", flush=True)
best_sha = sha256_file(best_path)
saved_decoder_source = RUN_DIR / "sources" / decoder_source.name
if saved_decoder_source.is_file():
    if sha256_file(saved_decoder_source) != sha256_file(decoder_source):
        raise ValueError("The saved run used a different decoder implementation.")
elif best_sha != known_decoder_sha:
    raise FileNotFoundError("For another decoder snapshot, attach the full run folder with sources/phase1_decoder.py.")

prompts, truths, groups = load_validation(PREP_DIR, SOURCE_ORDER)
evaluator_path = SRC_DIR / "evaluate_phase1.py"
evaluator_sha = sha256_file(evaluator_path)
if evaluator_sha != "3f3a9e62be718b478e3f52b378fcc535350ab90b9cb84e3a4c49fec247abf1db":
    raise ValueError("The course evaluator file differs from the supplied original.")

environment = {name: version(name) for name in ("torch", "bert-score", "transformers", "huggingface-hub")}
environment.update(python=platform.python_version(), cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0))
metric_model = lang2model["en"]
metric_defaults = {
    "model_type": metric_model, "layers": model2layers[metric_model],
    "batch_size": 8, "idf": False, "rescale_with_baseline": False,
}
protocol = {
    "run_id": RUN_ID,
    "checkpoint_sha256": best_sha,
    "decoder_source_sha256": sha256_file(decoder_source),
    "tokenizer_sha256": manifest["tokenizer_state_sha256"],
    "tokenizer_signature": tokenizer_signature,
    "validation_inputs_sha256": {
        name: sha256_file(PREP_DIR / f"{name}.json")
        for name in ("validation_prompts", "validation_truth", "validation_groups")
    },
    "generation_source_sha256": sha256_file(SRC_DIR / "phase1_generation.py"),
    "batch_size": GENERATION_BATCH_SIZE,
    "token_cap": MAX_NEW_TOKENS,
    "seed": GENERATION_SEED,
    "runtime": {name: environment[name] for name in ("python", "torch", "cuda", "gpu")},
}

evaluation_id = f"{RUN_ID}-{best_sha[:12]}-{EVALUATION_TAG}"
EVAL_DIR = OUTPUT_ROOT / evaluation_id
if EVAL_RESUME_INPUT is not None:
    resume = resolve_input(EVAL_RESUME_INPUT, (), "evaluation_protocol.json", "EVAL_RESUME_INPUT")
    if read_json(resume / "evaluation_protocol.json") != protocol:
        raise ValueError("The evaluation backup differs in inputs, generation settings or GPU runtime.")
    if not EVAL_DIR.exists():
        print("Restoring the saved evaluation folder", flush=True)
        restore_folder(resume, EVAL_DIR)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
checked_json(EVAL_DIR / "evaluation_protocol.json", protocol)
print(f"Checkpoint SHA-256: {best_sha}")
print(f"Evaluation folder: {EVAL_DIR}")
print(f"BERTScore scorer: {metric_model}")


Preparation folder: /kaggle/input/datasets/ash9201k/prep-2/cs781_phase1/preparation/prep_002_mixed16k
Decoder folder: /kaggle/input/datasets/ash9201k/decoder-004/cs781_phase1/runs/decoder_004_mixed217m
Hashing the saved decoder checkpoint
Checkpoint SHA-256: fd5add7ccc1d519a6f547fbd9a5c86a53276db8e09fb93c165d96531685d0bb7
Evaluation folder: /kaggle/working/phase1_decoder_evaluation_outputs/decoder_004_mixed217m-fd5add7ccc1d-v1
BERTScore scorer: roberta-large


## 4. Generate held-out continuations

Greedy selects the highest-probability token. Top-p sampling keeps the smallest set
covering 95% of the probability mass and samples within it, with temperature applied
before selection. Generation saves progress after each batch and prints a periodic
update. The token cap protects against runs that fail to complete a 128-word continuation.

The previous Phase 1 report records balanced BERTScore F1 values of 0.810716 for greedy,
0.812268 for top-p at temperature 0.1, and 0.839593 for top-p at temperature 1.0. The
table below reports this evaluation's measured results, which can differ with the
checkpoint or runtime.


In [6]:
# Load the tokenizer only from your own preparation backup.
with tokenizer_path.open("rb") as file:
    saved_tokenizer = pickle.load(file)
if saved_tokenizer["signature"] != tokenizer_signature:
    raise ValueError("Saved tokenizer signature differs from the preparation manifest.")
tokenizer = A1BPETokenizer()
tokenizer.__dict__.update(saved_tokenizer["state"])
if tokenizer.vocab_size != 16_000:
    raise ValueError("The decoder expects the 16,000-ID vocabulary.")


def load_decoder():
    print("Loading best validation-NLL weights", flush=True)
    best = torch.load(best_path, map_location="cpu", weights_only=True)
    if best["tokenizer_signature"] != tokenizer_signature:
        raise ValueError("The checkpoint was trained with a different tokenizer.")
    model = DecoderLM(LMConfig(**best["model_config"]))
    parameters = sum(parameter.numel() for parameter in model.parameters())
    if parameters != 217_860_096 or model.cfg.max_seq_len != 2560:
        raise ValueError("Expected the Phase 1 217.9M decoder with a 2,560-token context.")
    model.load_state_dict(best["model_state"], strict=True)
    return model.cuda().eval(), best["validation_result"], parameters


def generate_split(model, name, split, examples):
    decode_protocol = {
        **DECODING_SETTINGS[name],
        "checkpoint_sha256": best_sha,
        "generation_source_sha256": protocol["generation_source_sha256"],
    }
    print(f"{split} · {name} · {len(examples):,} prompts", flush=True)
    return generate_predictions(
        model, tokenizer, examples, EVAL_DIR / split / name, decode_protocol,
        batch_size=GENERATION_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
        target_words=128, seed=GENERATION_SEED,
    )


def incomplete_ids(rows):
    return [key for key, row in rows.items() if not row["complete"] or row["word_count"] != 128]


lm, selected_validation, parameters = load_decoder()
positions = selected_validation["tokens_in_applied_updates"]
nll = selected_validation["validation_nll"]
print(f"{parameters:,} parameters · {positions:,} training positions at selection · held-out NLL {nll:.6f}")
atomic_json(EVAL_DIR / "checkpoint_summary.json", {
    "parameters": parameters, "checkpoint_sha256": best_sha, "selection": selected_validation,
})
generation_rows = []
try:
    for name in RUN_DECODING_NAMES:
        predictions, diagnostics = generate_split(lm, name, "validation", prompts)
        failures = incomplete_ids(diagnostics)
        generation_rows.append({
            "decoding": name, "prompts": len(prompts), "complete_128_words": len(prompts) - len(failures),
            "incomplete": len(failures),
        })
finally:
    del lm
    gc.collect()
    torch.cuda.empty_cache()
display(pd.DataFrame(generation_rows))


Loading best validation-NLL weights
217,860,096 parameters · 300,011,520 training positions at selection · held-out NLL 1.387249
validation · greedy · 125 prompts
Generation · 125/125 saved prompts · batch size 4
validation · top_p_095_t01 · 125 prompts
Generation · 125/125 saved prompts · batch size 4
validation · top_p_095_t1 · 125 prompts
Generation · 125/125 saved prompts · batch size 4


,decoding,prompts,complete_128_words,incomplete
0,greedy,125,125,0
1,top_p_095_t01,125,125,0
2,top_p_095_t1,125,125,0


## 5. Score the frozen validation groups

Use the supplied evaluator on the balanced set, the original Supreme Court anchors,
and the four balanced source slices. The first scoring call may download the BERTScore
encoder. Scores are saved per slice and reused if the predictions, references, evaluator,
and metric runtime match. A decoding setting with unfinished continuations is listed in the
generation diagnostics and excluded from the score table.


In [7]:
import re
import subprocess

score_rows = []
scoring_groups = {
    "balanced_100": groups["balanced_100"],
    "anchor_50": groups["anchor_50"],
    **{f"source_{source}": ids for source, ids in groups["balanced_source_ids"].items()},
}
for name in RUN_DECODING_NAMES:
    generation_dir = EVAL_DIR / "validation" / name
    predictions = read_json(generation_dir / "prediction.json")
    failures = incomplete_ids(read_json(generation_dir / "progress.json")["rows"])
    if failures:
        print(f"{name} · {len(failures)} incomplete continuations · see {generation_dir / 'summary.json'}")
        continue
    print(f"Scoring {name} · {len(scoring_groups)} groups", flush=True)
    for group_name, ids in scoring_groups.items():
        score_dir = EVAL_DIR / "scores" / name / group_name
        score_dir.mkdir(parents=True, exist_ok=True)
        truth_path, prediction_path = score_dir / "truth.json", score_dir / "prediction.json"
        atomic_json(truth_path, {key: truths[key] for key in ids})
        atomic_json(prediction_path, {key: predictions[key] for key in ids})
        score_protocol = {
            "truth_sha256": sha256_file(truth_path),
            "prediction_sha256": sha256_file(prediction_path),
            "evaluator_sha256": evaluator_sha,
            "environment": environment,
            "metric_defaults": metric_defaults,
        }
        protocol_path = score_dir / "protocol.json"
        result_path = score_dir / "result.json"
        if result_path.is_file() and protocol_path.is_file() and read_json(protocol_path) == score_protocol:
            result = read_json(result_path)
        else:
            print(f"  {group_name} · {len(ids)} examples", flush=True)
            completed = subprocess.run(
                [sys.executable, str(evaluator_path), "--truth", str(truth_path), "--prediction", str(prediction_path)],
                capture_output=True, text=True,
            )
            (score_dir / "stdout.txt").write_text(completed.stdout, encoding="utf-8")
            (score_dir / "stderr.txt").write_text(completed.stderr, encoding="utf-8")
            if completed.returncode:
                raise RuntimeError(f"Course evaluator failed for {name}/{group_name}:\n{completed.stderr[-4000:]}")
            result = {"decoding": name, "group": group_name, "examples": len(ids)}
            for label, key in (("BERTScore F1", "bertscore_f1"), ("ROUGE-L F1", "rouge_l_f1"), ("BLEU-4", "bleu4")):
                match = re.search(re.escape(label) + r":\s*([0-9.eE+-]+)", completed.stdout)
                if not match:
                    raise ValueError(f"Missing {label} in the evaluator output.")
                result[key] = float(match.group(1))
            result_path.unlink(missing_ok=True)
            atomic_json(protocol_path, score_protocol)
            atomic_json(result_path, result)
        score_rows.append(result)

score_summary_path = EVAL_DIR / "score_summary.json"
atomic_json(score_summary_path, score_rows)
if score_rows:
    scores = pd.DataFrame(score_rows)
    with pd.option_context("display.precision", 6):
        display(scores)
    balanced_scores = scores[scores["group"] == "balanced_100"].sort_values("bertscore_f1", ascending=False)
    strongest = balanced_scores.iloc[0]
    print(f"Best balanced BERTScore F1 · {strongest['decoding']} · {strongest['bertscore_f1']:.6f}")
else:
    print("No setting completed all held-out continuations.")
show_download(score_summary_path)


Scoring greedy · 6 groups
Scoring top_p_095_t01 · 6 groups
Scoring top_p_095_t1 · 6 groups


,decoding,group,examples,bertscore_f1,rouge_l_f1,bleu4
0,greedy,balanced_100,100,0.810716,0.244688,0.132353
1,greedy,anchor_50,50,0.786609,0.169844,0.026111
2,greedy,source_supreme_court,25,0.781648,0.155312,0.023651
3,greedy,source_high_courts,25,0.863781,0.438750,0.369201
4,greedy,source_acts,25,0.789167,0.180312,0.041049
5,greedy,source_tribunals,25,0.808267,0.204375,0.042419
6,top_p_095_t01,balanced_100,100,0.812268,0.247812,0.131893
7,top_p_095_t01,anchor_50,50,0.788515,0.171094,0.027394
8,top_p_095_t01,source_supreme_court,25,0.783222,0.162188,0.022600
9,top_p_095_t01,source_high_courts,25,0.866584,0.441250,0.367634


Best balanced BERTScore F1 · top_p_095_t1 · 0.839593


/kaggle/working/phase1_decoder_evaluation_outputs/decoder_004_mixed217m-fd5add7ccc1d-v1/score_summary.json

## 6. Optional official submission export

Choose a setting with a saved balanced validation score and enable
`GENERATE_OFFICIAL_TEST` in section 1. Generation may take a while across 1,000 prompts,
especially on a T4. Keep the same evaluation settings to resume from completed batches.
The export checks every ID and continuation length before writing the ZIP. The notebook
generates the file for manual submission to CodaBench.


In [8]:
import zipfile

if GENERATE_OFFICIAL_TEST:
    score_file = EVAL_DIR / "scores" / TEST_DECODING_NAME / "balanced_100" / "result.json"
    validation_dir = EVAL_DIR / "validation" / TEST_DECODING_NAME
    score_protocol_path = score_file.with_name("protocol.json")
    if (
        not score_file.is_file()
        or not score_protocol_path.is_file()
        or not (validation_dir / "prediction.json").is_file()
        or incomplete_ids(read_json(validation_dir / "progress.json")["rows"])
    ):
        raise ValueError("Run validation and scoring for TEST_DECODING_NAME before exporting.")
    previous_protocol = read_json(validation_dir / "protocol.json")
    if any(previous_protocol.get(key) != value for key, value in DECODING_SETTINGS[TEST_DECODING_NAME].items()):
        raise ValueError("The saved validation used different decoding settings. Re-evaluate before exporting.")

    official_prompts = load_test_prompts(PREP_DIR)
    lm, _, _ = load_decoder()
    try:
        test_predictions, test_rows = generate_split(lm, TEST_DECODING_NAME, "official_test", official_prompts)
    finally:
        del lm
        gc.collect()
        torch.cuda.empty_cache()
    if set(test_predictions) != set(official_prompts) or set(test_rows) != set(official_prompts):
        raise ValueError("Official prediction IDs differ from the official prompt IDs.")
    failures = set(incomplete_ids(test_rows))
    failures.update(key for key, row in test_predictions.items() if len(row["text"].split()) != 128)
    if failures:
        raise ValueError(f"Official predictions contain {len(failures)} unfinished or wrong-length continuations.")
    test_dir = EVAL_DIR / "official_test" / TEST_DECODING_NAME
    submission_path = test_dir / f"phase1_decoder_{TEST_DECODING_NAME}_submission.zip"
    temporary = submission_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(test_dir / "prediction.json", "prediction.json")
    temporary.replace(submission_path)
    print(f"Submission ready · {len(test_predictions):,} continuations")
    show_download(submission_path)
else:
    print("Official test generation is disabled. Review validation scores before exporting.")


Official test generation is disabled. Review validation scores before exporting.


## 7. Save the evaluation backup

The archive holds evaluation settings, generated predictions, score tables, and resumable batch
progress. You can run this cell separately after stopping a long generation stage. Save the ZIP
privately. To resume, attach the extracted folder and use `EVAL_RESUME_INPUT`.


In [9]:
if CREATE_EVALUATION_ZIP:
    backup_path = OUTPUT_ROOT / f"{evaluation_id}_backup.zip"
    print("Writing evaluation backup", flush=True)
    make_bundle(EVAL_DIR, backup_path, OUTPUT_ROOT)
    show_download(backup_path)


Writing evaluation backup


/kaggle/working/phase1_decoder_evaluation_outputs/decoder_004_mixed217m-fd5add7ccc1d-v1_backup.zip

## Provenance

The decoder weights come from the Phase 1 decoder training run. The preparation notebook
documents the shared tokenizer and validation split. The exact course evaluator is embedded
in section 2 and its SHA-256 is saved with the scores.
